# A/B Hypothesis Testing
In this notebook, we statistically validate key hypotheses about risk drivers to inform segmentation and pricing.


In [1]:
import pandas as pd
import sys
sys.path.append('../src')
from hypothesis_tests import test_categorical_kpi, test_numerical_kpi_multiple, test_numerical_kpi_two

df = pd.read_csv('../data/insurance_data.csv')

# KPIs
df['ClaimFrequencyFlag'] = (df['TotalClaims'] > 0).astype(int)
df['ClaimSeverity'] = df['TotalClaims']  # For records where TotalClaims > 0, which is most
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

results = []


In [2]:
# H1: There are no risk differences across provinces. KPI: ClaimSeverity (ANOVA)
p_val, decision = test_numerical_kpi_multiple(df, 'Province', 'ClaimSeverity')
results.append({'Hypothesis': 'Provinces risk differences', 'Test': 'ANOVA', 'P-Value': p_val, 'Decision': decision})
print(f"H1 (Provinces vs Severity): p-value = {p_val:.4f} -> {decision}")


H1 (Provinces vs Severity): p-value = 0.0171 -> Reject H0


In [3]:
# H2: There are no risk differences between zip codes. KPI: ClaimFrequencyFlag (Chi-Squared)
p_val, decision = test_categorical_kpi(df, 'ZipCode', 'ClaimFrequencyFlag')
results.append({'Hypothesis': 'ZipCode risk differences', 'Test': 'Chi-Squared', 'P-Value': p_val, 'Decision': decision})
print(f"H2 (ZipCode vs Frequency): p-value = {p_val:.4f} -> {decision}")


H2 (ZipCode vs Frequency): p-value = 1.0000 -> Fail to reject H0


In [4]:
# H3: There is no significant margin (profit) difference between zip codes. KPI: Margin (ANOVA)
p_val, decision = test_numerical_kpi_multiple(df, 'ZipCode', 'Margin')
results.append({'Hypothesis': 'ZipCode margin differences', 'Test': 'ANOVA', 'P-Value': p_val, 'Decision': decision})
print(f"H3 (ZipCode vs Margin): p-value = {p_val:.4f} -> {decision}")


H3 (ZipCode vs Margin): p-value = nan -> Fail to reject H0


C:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_axis_nan_policy.py:586: SmallSampleWarning: all input arrays have length 1.  f_oneway requires that at least one input has length greater than 1.
  if is_too_small(samples, kwds):
C:\Users\HP\tenx\week3\insurance-risk-analytics\notebooks\../src\hypothesis_tests.py:22: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  F, p_val = f_oneway(*groups)


In [5]:
# H4: There is no significant risk difference between Women and Men. KPI: ClaimSeverity (T-Test)
# Filter strictly Men and Women
df_gender = df[df['Gender'].isin(['M', 'F'])].copy()
p_val, decision = test_numerical_kpi_two(df_gender, 'Gender', 'ClaimSeverity')
results.append({'Hypothesis': 'Gender risk differences', 'Test': 'T-Test', 'P-Value': p_val, 'Decision': decision})
print(f"H4 (Gender vs Severity): p-value = {p_val:.4f} -> {decision}")


H4 (Gender vs Severity): p-value = 0.5375 -> Fail to reject H0


In [6]:
import matplotlib.pyplot as plt
results_df = pd.DataFrame(results)
display(results_df)

print("\n--- Business Interpretations ---")
for _, row in results_df.iterrows():
    if row['Decision'] == 'Reject H0':
        print(f"[REJECT H0]: We reject the null hypothesis for {row['Hypothesis']} (p = {row['P-Value']:.4f}). This indicates significant differences across this dimension, suggesting a targeted adjustment to our premiums is warranted for distinct subsets in this group.")
    else:
        print(f"[FAIL TO REJECT H0]: We fail to reject the null hypothesis for {row['Hypothesis']} (p = {row['P-Value']:.4f}). There is insufficient statistical evidence to say the risk varies, meaning our baseline pricing covers the underlying variance.")


,Hypothesis,Test,P-Value,Decision
0,Provinces risk differences,ANOVA,0.017095,Reject H0
1,ZipCode risk differences,Chi-Squared,1.000000,Fail to reject H0
2,ZipCode margin differences,ANOVA,NaN,Fail to reject H0
3,Gender risk differences,T-Test,0.537510,Fail to reject H0



--- Business Interpretations ---
[REJECT H0]: We reject the null hypothesis for Provinces risk differences (p = 0.0171). This indicates significant differences across this dimension, suggesting a targeted adjustment to our premiums is warranted for distinct subsets in this group.
[FAIL TO REJECT H0]: We fail to reject the null hypothesis for ZipCode risk differences (p = 1.0000). There is insufficient statistical evidence to say the risk varies, meaning our baseline pricing covers the underlying variance.
[FAIL TO REJECT H0]: We fail to reject the null hypothesis for ZipCode margin differences (p = nan). There is insufficient statistical evidence to say the risk varies, meaning our baseline pricing covers the underlying variance.
[FAIL TO REJECT H0]: We fail to reject the null hypothesis for Gender risk differences (p = 0.5375). There is insufficient statistical evidence to say the risk varies, meaning our baseline pricing covers the underlying variance.
